# Sample composition (v0.4)

Mix real audio samples (drum loops, synth pads, bass one-shots) alongside or instead of FluidSynth-rendered layers.

**Requires:** `pip install -e '.[samples]'` (installs `audio-sample-manager`, `soundfile`, `rubberband-stretch` on Linux/macOS) plus the base FluidSynth dependency.

**Cell tags:**
- `# requires: samples` — needs `musicgen[samples]` extra
- `# requires: fluidsynth` — needs FluidSynth binary
- `# requires: wavs` — needs a directory of real audio sample WAVs

## 0. Cloud setup (Colab / Binder)

Run this cell first when launching on Google Colab or mybinder.org. On a local install it's a no-op.

In [ ]:
# Colab/Binder bootstrap — skipped on local installs
import sys, os, shutil

IN_COLAB = "google.colab" in sys.modules
IN_BINDER = "JUPYTERHUB_USER" in os.environ or os.path.exists("/home/jovyan")

if IN_COLAB:
    # apt deps (fluidsynth binary + FluidR3_GM soundfont)
    !apt-get -qq install -y fluidsynth fluid-soundfont-gm ffmpeg
    # musicgen + neural extra
    !pip install -q "musicgen[neural] @ git+https://github.com/dobidu/layered_music_gen.git"
    # Clone repo for config.py + genres/ + notebooks (sys.path needs repo root)
    if not os.path.exists("/content/musicgen_repo"):
        !git clone -q https://github.com/dobidu/layered_music_gen.git /content/musicgen_repo
    os.chdir("/content/musicgen_repo")
    # Link the soundfont into per-layer sf dirs
    SF2 = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
    for layer in ("beat", "melody", "harmony", "bassline"):
        os.makedirs(f"sf/{layer}", exist_ok=True)
        dst = f"sf/{layer}/FluidR3_GM.sf2"
        if not os.path.exists(dst):
            os.symlink(SF2, dst)
    print("Colab setup complete. Repo root:", os.getcwd())
elif IN_BINDER:
    print("Binder env detected — apt deps + soundfonts handled by binder/postBuild")
else:
    print("Local env — assumed to have FluidSynth + .sf2 files installed already")


## 1. Setup

In [ ]:
import shutil, sys, os, tempfile, json, pathlib

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from config import Config
from musicgen import __version__

HAS_FLUIDSYNTH = shutil.which("fluidsynth") is not None

try:
    import audio_sample_manager  # noqa
    HAS_SAMPLES = True
except ImportError:
    HAS_SAMPLES = False

try:
    import musicality  # noqa
    HAS_MUSICALITY = True
except ImportError:
    HAS_MUSICALITY = False

print(f"musicgen {__version__}")
print(f"FluidSynth present: {HAS_FLUIDSYNTH}")
print(f"audio_sample_manager: {HAS_SAMPLES}")
print(f"musicality standalone: {HAS_MUSICALITY}")

## 2. Build a sample library (M5)

`musicgen samples build` walks a directory of audio files, annotates each with `audio_sample_manager`, and writes a `SampleManager` JSON database.

For this demo we synthesize 3 short sine-wave WAVs to act as fake "samples".

In [ ]:
# requires: samples
import numpy as np
import soundfile as sf

def write_sine(path, freq_hz, duration_s=2.0, sr=44100, bpm=120.0):
    t = np.linspace(0, duration_s, int(sr * duration_s), endpoint=False)
    wav = 0.3 * np.sin(2 * np.pi * freq_hz * t).astype(np.float32)
    sf.write(path, wav, sr)

if not HAS_SAMPLES:
    print("Skipping: audio_sample_manager not installed")
else:
    samples_dir = tempfile.mkdtemp(prefix="musicgen_samples_")
    # Filenames carry category keywords so build auto-detects category
    write_sine(os.path.join(samples_dir, "kick_120bpm.wav"),  60.0)
    write_sine(os.path.join(samples_dir, "bass_120bpm.wav"), 120.0)
    write_sine(os.path.join(samples_dir, "pad_C_chord.wav"), 261.6)
    print(f"Wrote 3 demo samples to {samples_dir}")

In [ ]:
# requires: samples
# Build the library JSON via the Python API (mirrors `musicgen samples build`)
if not HAS_SAMPLES:
    print("Skipping: audio_sample_manager not installed")
else:
    from musicgen.sample_builder import build_library

    lib_path = os.path.join(samples_dir, "library.json")
    n = build_library(
        wav_dir=samples_dir,
        output=lib_path,
        category_override=None,        # auto-detect from filename keywords
        compute_musicality=False,
        recursive=False,
    )
    print(f"Built library with {n} samples → {lib_path}")
    print("\nLibrary entries:")
    db = json.loads(pathlib.Path(lib_path).read_text())
    for entry in db.get("samples", [])[:5]:
        print(f"  {entry.get('name')}  cat={entry.get('category')}  bpm={entry.get('bpm')}")

## 3. SampleCompositionConfig + per-layer rules (M3)

`SampleLayerRule` configures one layer (beat / bassline / melody / harmony). `SampleCompositionConfig` aggregates rules + global settings (transposition toggle, BPM stretch, min musicality).

In [ ]:
# requires: samples
if not HAS_SAMPLES:
    print("Skipping: audio_sample_manager not installed")
else:
    from musicgen.sample_composition import SampleLayerRule, SampleCompositionConfig

    sc_cfg = SampleCompositionConfig(
        sample_db_path=lib_path,
        layer_rules={
            "beat": SampleLayerRule(
                layer="beat",
                mode="alongside",          # overlaid on FluidSynth mix
                gain_db=-6.0,
                max_bpm_stretch_pct=20.0,  # reject samples needing > 20% stretch
            ),
            "bassline": SampleLayerRule(
                layer="bassline",
                mode="substitution",       # replaces FluidSynth stem
                gain_db=-3.0,
            ),
        },
        global_min_musicality=None,
        allow_transposition=True,
        allow_time_stretching=True,
    )
    print(f"sample_db_path: {sc_cfg.sample_db_path}")
    print(f"active layers:  {list(sc_cfg.layer_rules)}")
    for layer, rule in sc_cfg.layer_rules.items():
        print(f"  {layer}: mode={rule.mode}, gain_db={rule.gain_db}")

## 4. Generate with sample composition (M3 + M4 + M5)

`sample.json` will gain a `used_samples` key listing which real audio was mixed in per layer.

In [ ]:
# requires: fluidsynth + samples
if not HAS_FLUIDSYNTH or not HAS_SAMPLES:
    print("Skipping: requires FluidSynth + musicgen[samples]")
else:
    from musicgen import generate

    with tempfile.TemporaryDirectory(prefix="musicgen_sc_") as out:
        cfg = Config.load(cli_overrides={
            "global_seed": 42,
            "sample_index": 0,
            "dataset_root": out,
            "sample_composition": sc_cfg,
        })
        result = generate(cfg)
        print(f"status: {result.status}")
        print(f"musicality: {result.musicality_score:.3f}")

        sj = json.loads(pathlib.Path(result.sample_json_path).read_text())
        used = sj.get("used_samples", {})
        print(f"\nused_samples in sample.json:")
        for layer, info in used.items():
            print(f"  {layer}: name={info['name']}  mode={info['mode']}  bpm={info['bpm']}")

## 5. Mixing modes side-by-side

Same seed, three different modes. Inspect `used_samples` to see how each behaves.

In [ ]:
# requires: fluidsynth + samples
if not HAS_FLUIDSYNTH or not HAS_SAMPLES:
    print("Skipping: requires FluidSynth + musicgen[samples]")
else:
    from musicgen import generate
    from musicgen.sample_composition import SampleLayerRule, SampleCompositionConfig

    for mode in ("alongside", "substitution"):
        with tempfile.TemporaryDirectory() as out:
            mode_cfg = SampleCompositionConfig(
                sample_db_path=lib_path,
                layer_rules={
                    "beat": SampleLayerRule(layer="beat", mode=mode, gain_db=-6.0),
                },
            )
            cfg = Config.load(cli_overrides={
                "global_seed": 7,
                "sample_index": 0,
                "dataset_root": out,
                "sample_composition": mode_cfg,
            })
            r = generate(cfg)
            sj = json.loads(pathlib.Path(r.sample_json_path).read_text())
            beat_info = sj.get("used_samples", {}).get("beat", {})
            print(f"mode={mode:<14} used beat sample: {beat_info.get('name', '(none)')}")

## 6. Quality gate — `min_musicality_score`

Per-layer or global threshold filters out samples scoring below the bound. Requires `--musicality` annotation during library build.

In [ ]:
# requires: samples + musicality
if not (HAS_SAMPLES and HAS_MUSICALITY):
    print("Skipping: requires audio_sample_manager + musicality")
else:
    # Rebuild library with musicality scoring
    from musicgen.sample_builder import build_library
    lib_q_path = os.path.join(samples_dir, "library_scored.json")
    n = build_library(
        wav_dir=samples_dir,
        output=lib_q_path,
        compute_musicality=True,   # score each sample
    )
    db = json.loads(pathlib.Path(lib_q_path).read_text())
    print(f"Scored {n} samples:")
    for s in db.get("samples", []):
        score = s.get("musicality_score")
        print(f"  {s.get('name'):<24} musicality_score={score}")

## 7. Standalone `musicality` package (M1)

`musicality` was extracted as an installable standalone package. It can score any WAV without needing the rest of musicgen.

In [ ]:
# requires: musicality
if not HAS_MUSICALITY:
    print("Skipping: musicality not installed")
else:
    import numpy as np
    import soundfile as sf
    from musicality import score, explain

    sr = 22050
    t = np.linspace(0, 2.0, int(sr * 2.0), endpoint=False)
    wav = 0.3 * np.sin(2 * np.pi * 261.6 * t).astype(np.float32)
    standalone_dir = tempfile.mkdtemp(prefix="musicality_demo_")
    standalone_wav = os.path.join(standalone_dir, "demo.wav")
    sf.write(standalone_wav, wav, sr)

    s = score(standalone_wav)
    print(f"score({os.path.basename(standalone_wav)}) = {s:.3f}")

    info = explain(standalone_wav)
    print("\nexplain() returned keys:")
    for k, v in info.items():
        print(f"  {k}: {v}")


## 8. Cleanup

In [ ]:
import shutil
for varname in ("samples_dir", "standalone_dir"):
    path = globals().get(varname)
    if path and os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)
        print(f"Removed {path}")


## Further reading

- [`docs/sample-composition.md`](../docs/sample-composition.md) — full reference
- [`docs/musicality-scoring.md`](../docs/musicality-scoring.md) — scoring theory and components
- README section: **Sample composition (v0.4)**